# **A. MEMBANGUN CASE BASE**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install pdfminer.six
import os, re
from pdfminer.high_level import extract_text

input_folder = "/content/drive/MyDrive/CBR_Putusan_Pemalsuan/dokumen_pdf"
output_folder = "/content/drive/MyDrive/CBR_Putusan_Pemalsuan/dokumen_raw"
os.makedirs(output_folder, exist_ok=True)

def clean_putusan(text):
    text = text.lower()
    text = re.sub(r'mahkamah agung republik indonesia', '', text)
    text = re.sub(r'direktori putusan.*?mahkamahagung\.go\.id', '', text)
    text = re.sub(r'disclaimer.*?hubungi.*?(email|telp).*?', '', text)
    text = re.sub(r'halaman\s*\d+\s*dari\s*\d+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

for i, filename in enumerate(sorted(os.listdir(input_folder))):
    if filename.endswith(".pdf"):
        text = extract_text(os.path.join(input_folder, filename))
        cleaned = clean_putusan(text)
        with open(os.path.join(output_folder, f"case_{i+1:03}.txt"), "w") as f:
            f.write(cleaned)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 40.9 MB/s eta 0:00:00


In [ ]:
!pip install pdfminer.six
import os, re
from pdfminer.high_level import extract_text

input_folder = "/content/drive/MyDrive/CBR_Putusan_Pemalsuan/dokumen_pdf"
output_folder = "/content/drive/MyDrive/CBR_Putusan_Pemalsuan/dokumen_raw"
log_path = "/content/drive/MyDrive/CBR_Putusan_Pemalsuan/logs/cleaning.log"
os.makedirs(output_folder, exist_ok=True)
os.makedirs(os.path.dirname(log_path), exist_ok=True)

def clean_text(text):
    text = text.lower()
    text = re.sub(r'mahkamah agung republik indonesia', '', text)
    text = re.sub(r'direktori putusan.*?mahkamahagung\.go\.id', '', text)
    text = re.sub(r'disclaimer.*?hubungi.*?(email|telp).*?', '', text)
    text = re.sub(r'halaman\s*\d+\s*dari\s*\d+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

with open(log_path, "w") as log:
    for i, filename in enumerate(sorted(os.listdir(input_folder))):
        if filename.endswith(".pdf"):
            text = extract_text(os.path.join(input_folder, filename))
            if len(text) < 1000:
                log.write(f"{filename} SKIPPED: too short\n")
                continue
            cleaned = clean_text(text)
            fpath = os.path.join(output_folder, f"case_{i+1:03}.txt")
            with open(fpath, "w") as f:
                f.write(cleaned)
            log.write(f"{filename} OK\n")


# **B. CASE REPRESENTATION**

In [ ]:
import pandas as pd

raw_folder = "/content/drive/MyDrive/CBR_Putusan_Pemalsuan/dokumen_raw"
processed_csv = "/content/drive/MyDrive/CBR_Putusan_Pemalsuan/processed/cases.csv"
os.makedirs(os.path.dirname(processed_csv), exist_ok=True)

cases = []
def extract_metadata(text):
    no_perkara = re.search(r'nomor[:\s]*([0-9./a-zA-Z-]+)', text)
    tanggal = re.search(r'tanggal[:\s]*([0-9]{1,2}\s\w+\s[0-9]{4})', text)
    pasal = re.findall(r'pasal\s+\d+(?:\s+ayat\s+\(\d+\))?', text)
    pihak = re.search(r'nama[:\s]*([a-z\s]+);', text)
    fakta = ". ".join(text.split(". ")[:3])
    argumen_match = re.findall(r'putusan.*?\.', text)
    argumen = " ".join(argumen_match[:2]) if argumen_match else ""
    return {
        "no_perkara": no_perkara.group(1) if no_perkara else "",
        "tanggal": tanggal.group(1) if tanggal else "",
        "pasal": "; ".join(pasal),
        "pihak": pihak.group(1).strip() if pihak else "",
        "ringkasan_fakta": fakta,
        "argumen_hukum": argumen
    }

for i, fname in enumerate(sorted(os.listdir(raw_folder))):
    if fname.endswith(".txt"):
        with open(os.path.join(raw_folder, fname), "r") as f:
            text = f.read()
            meta = extract_metadata(text)
            meta["case_id"] = f"case_{i+1:03}"
            meta["text_full"] = text[:300]
            cases.append(meta)

df = pd.DataFrame(cases)
df.to_csv(processed_csv, index=False)


In [ ]:
import pandas as pd

raw_folder = output_folder
processed_path = "/content/drive/MyDrive/CBR_Putusan_Pemalsuan/processed/cases.csv"
os.makedirs(os.path.dirname(processed_path), exist_ok=True)

cases = []
for i, fname in enumerate(sorted(os.listdir(raw_folder))):
    if fname.endswith(".txt"):
        with open(os.path.join(raw_folder, fname), "r") as f:
            text = f.read()
        meta = {
            "case_id": f"case_{i+1:03}",
            "no_perkara": re.search(r'nomor[:\s]*([0-9./a-zA-Z-]+)', text).group(1) if re.search(r'nomor[:\s]*([0-9./a-zA-Z-]+)', text) else "",
            "tanggal": re.search(r'tanggal[:\s]*([0-9]{1,2}\s\w+\s[0-9]{4})', text).group(1) if re.search(r'tanggal[:\s]*([0-9]{1,2}\s\w+\s[0-9]{4})', text) else "",
            "pasal": "; ".join(re.findall(r'pasal\s+\d+(?:\s+ayat\s+\(\d+\))?', text)),
            "pihak": re.search(r'nama[:\s]*([a-z\s]+);', text).group(1) if re.search(r'nama[:\s]*([a-z\s]+);', text) else "",
            "ringkasan_fakta": ". ".join(text.split(". ")[:3]),
            "argumen_hukum": " ".join(re.findall(r'putusan.*?\.', text)[:2]),
            "text_full": text[:300]
        }
        cases.append(meta)

pd.DataFrame(cases).to_csv(processed_path, index=False)


# **C. CASE RETRIEVAL (TF-IDF)**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import json

df = pd.read_csv(processed_csv)
documents = df["ringkasan_fakta"].fillna("").tolist()
case_ids = df["case_id"].tolist()

vectorizer = TfidfVectorizer()
doc_vectors = vectorizer.fit_transform(documents)

def retrieve(query: str, k: int = 5):
    qvec = vectorizer.transform([query])
    sims = cosine_similarity(qvec, doc_vectors).flatten()
    topk = sims.argsort()[::-1][:k]
    return [case_ids[i] for i in topk]

# Simpan contoh query
queries = [
    {"query_id": "q1", "query": "terdakwa memalsukan sertifikat tanah"},
    {"query_id": "q2", "query": "pemalsuan dokumen kependudukan"},
]
eval_path = "/content/drive/MyDrive/CBR_Putusan_Pemalsuan/eval/queries.json"
os.makedirs(os.path.dirname(eval_path), exist_ok=True)
with open(eval_path, "w") as f:
    json.dump(queries, f, indent=2)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import json

df = pd.read_csv(processed_path)
docs = df["ringkasan_fakta"].fillna("").tolist()
case_ids = df["case_id"].tolist()
vectorizer = TfidfVectorizer()
doc_vecs = vectorizer.fit_transform(docs)

def retrieve(query: str, k: int = 5):
    qvec = vectorizer.transform([query])
    scores = cosine_similarity(qvec, doc_vecs).flatten()
    topk = scores.argsort()[::-1][:k]
    return [case_ids[i] for i in topk]

queries = [
    {"query_id": "q1", "query": "terdakwa memalsukan sertifikat tanah"},
    {"query_id": "q2", "query": "penggugat menggugat karena akta hibah tidak sah"},
    {"query_id": "q3", "query": "pemalsuan dokumen untuk pinjaman kredit"}
]
with open("/content/drive/MyDrive/CBR_Putusan_Pemalsuan/eval/queries.json", "w") as f:
    json.dump(queries, f, indent=2)


# **D. SOLUTION REUSE**

In [ ]:
# Buat contoh dictionary case_solutions (isi: amar putusan)
case_solutions = {row["case_id"]: row["argumen_hukum"] for _, row in df.iterrows()}

def predict_outcome(query: str):
    top_k = retrieve(query, k=5)
    solutions = [case_solutions[c] for c in top_k if case_solutions[c]]
    if not solutions:
        return ""
    return max(set(solutions), key=solutions.count)  # majority vote

results = []
for q in queries:
    pred = predict_outcome(q["query"])
    results.append({
        "query_id": q["query_id"],
        "predicted_solution": pred,
        "top_5_case_ids": retrieve(q["query"])
    })

pred_path = "/content/drive/MyDrive/CBR_Putusan_Pemalsuan/results/predictions.csv"
os.makedirs(os.path.dirname(pred_path), exist_ok=True)
pd.DataFrame(results).to_csv(pred_path, index=False)


In [ ]:
solutions = {row["case_id"]: row["argumen_hukum"] for _, row in df.iterrows()}

def predict_outcome(query: str):
    top_k = retrieve(query, k=5)
    sols = [solutions[c] for c in top_k if solutions[c]]
    return max(set(sols), key=sols.count) if sols else ""

results = []
for q in queries:
    results.append({
        "query_id": q["query_id"],
        "predicted_solution": predict_outcome(q["query"]),
        "top_5_case_ids": retrieve(q["query"])
    })

pd.DataFrame(results).to_csv("/content/drive/MyDrive/CBR_Putusan_Pemalsuan/results/predictions.csv", index=False)


# **E. EVALUASI MODEL**

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# Dummy evaluation (jika punya ground truth)
# Misalnya kamu buat list label ground-truth hasil putusan dan hasil prediksi
y_true = ["bersalah", "bersalah"]
y_pred = ["bersalah", "tidak bersalah"]

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
rec = recall_score(y_true, y_pred, average='macro', zero_division=0)
f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

df_eval = pd.DataFrame([{"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}])
df_eval.to_csv("/content/drive/MyDrive/CBR_Putusan_Pemalsuan/eval/retrieval_metrics.csv", index=False)


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Dummy label
gt = pd.DataFrame({
    "query_id": ["q1", "q2", "q3"],
    "ground_truth": ["bersalah", "bersalah", "tidak bersalah"]
})
gt.to_csv("/content/drive/MyDrive/CBR_Putusan_Pemalsuan/eval/ground_truth.csv", index=False)

# Prediksi (kamu bisa update kalau ada label sebenarnya)
preds = pd.read_csv("/content/drive/MyDrive/CBR_Putusan_Pemalsuan/results/predictions.csv")
preds["ground_truth"] = gt["ground_truth"]
preds["predicted_label"] = ["bersalah", "tidak bersalah", "tidak bersalah"]  # contoh dummy

acc = accuracy_score(preds["ground_truth"], preds["predicted_label"])
prec = precision_score(preds["ground_truth"], preds["predicted_label"], average='macro', zero_division=0)
rec = recall_score(preds["ground_truth"], preds["predicted_label"], average='macro', zero_division=0)
f1 = f1_score(preds["ground_truth"], preds["predicted_label"], average='macro', zero_division=0)

pd.DataFrame([{
    "accuracy": acc, "precision": prec, "recall": rec, "f1": f1
}]).to_csv("/content/drive/MyDrive/CBR_Putusan_Pemalsuan/eval/prediction_metrics.csv", index=False)
